# E05 — quando as coisas ruins vêm juntas

Os capítulos anteriores mediram uma série por vez: o corte, o vigia, a célula do calendário.
Todos leem **uma** margem. Este caderno mede o que nenhum deles pode ver.

**A tentativa.** Proteger cada mercado no seu próprio corte de 5% e supor que a proteção
conjunta é a soma das duas.

**O que se mede.**

1. quantos dias os **dois** rompem o próprio corte, contra o que a independência previria;
2. o mesmo par com o segundo deslocado — a margem intacta, o par trocado, e quanto isso muda;
3. o controle: um par sorteado independente, onde a razão tem de dar um;
4. a perda da carteira no dia conjunto, contra o dia em que só uma perna rompe.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E05_dependencia.json, figura em .pdf e .png.

In [1]:
# <- brinque com: SERIE_A, SERIE_B, JANELA, CAUDA, ATRASOS, PESO, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, dependencia, graficos, volatilidade

RAIZ = Path.cwd()
SERIE_A = "sp500.csv"        # a primeira perna
SERIE_B = "ibov.csv"         # a segunda perna
JANELA = 252                 # o corte de cada perna, como no capitulo 1
CAUDA = 0.05
ATRASOS = (1, 21, 260, 520)  # um dia, um mes, um ano, dois anos
PESO = 0.5
SEMENTE = 53

retornos_a = volatilidade.retornos_log(dados.carregar_serie(SERIE_A))
retornos_b = volatilidade.retornos_log(dados.carregar_serie(SERIE_B))
rompe_a = dependencia.rompimentos(retornos_a, JANELA, CAUDA)
rompe_b = dependencia.rompimentos(retornos_b, JANELA, CAUDA)
print("frevolab %s | %s: %d dias | %s: %d dias | comuns: %d" % (
    frevolab.VERSAO, SERIE_A, len(retornos_a), SERIE_B, len(retornos_b),
    len(retornos_a.index.intersection(retornos_b.index))))

frevolab 0.1.0 | sp500.csv: 6718 dias | ibov.csv: 6620 dias | comuns: 6450


## Os dois cortes, cada um no seu mercado

In [2]:
# O mesmo corte em cada perna, e a conta conjunta contra o que a independencia previria.
par = dependencia.juntos(rompe_a, rompe_b)
print("dias %d | rompe %s: %d (%.3f%%) | rompe %s: %d (%.3f%%)" % (
    par["dias"], SERIE_A, par["rompe_a"], 100 * par["taxa_a"],
    SERIE_B, par["rompe_b"], 100 * par["taxa_b"]))
print("rompem JUNTOS: %d dias = %.3f%% | se fossem independentes: %.3f%% = %.1f dias" % (
    par["juntos"], 100 * par["taxa_juntos"], 100 * par["esperado"], par["esperado_dias"]))
print("quantas vezes mais que a independencia: %.2fx" % par["excesso"])
print("dos rompimentos de %s, quantos vem acompanhados: %.1f%% (independente: %.1f%%)" % (
    SERIE_A, 100 * par["acompanhados"], 100 * par["taxa_b"]))

dias 6204 | rompe sp500.csv: 315 (5.077%) | rompe ibov.csv: 316 (5.093%)
rompem JUNTOS: 115 dias = 1.854% | se fossem independentes: 0.259% = 16.0 dias
quantas vezes mais que a independencia: 7.17x
dos rompimentos de sp500.csv, quantos vem acompanhados: 36.5% (independente: 5.1%)


## O par é que decide

In [3]:
# O par e que decide: a segunda perna deslocada, e a margem das duas intacta.
linhas = []
for atraso in (0,) + ATRASOS:
    b_deslocado = dependencia.pareado(retornos_b, atraso)
    rompe_b_deslocado = dependencia.rompimentos(b_deslocado, JANELA, CAUDA)
    medida = dependencia.juntos(rompe_a, rompe_b_deslocado)
    comuns = medida and rompe_a.index.intersection(rompe_b_deslocado.index)
    corr = float(np.corrcoef(retornos_a.loc[comuns], b_deslocado.loc[comuns])[0, 1])
    linhas.append({"atraso": atraso, "rompe a": medida["rompe_a"], "rompe b": medida["rompe_b"],
                   "juntos": medida["juntos"], "juntos (%)": 100 * medida["taxa_juntos"],
                   "x independencia": medida["excesso"], "correlacao": corr})
tabela = pd.DataFrame(linhas).set_index("atraso")
print(tabela.round(3).to_string())

        rompe a  rompe b  juntos  juntos (%)  x independencia  correlacao
atraso                                                                   
0           315      316     115       1.854            7.168       0.589
1           315      317      23       0.371            1.429      -0.081
21          315      313      19       0.306            1.196      -0.032
260         315      324      11       0.177            0.669      -0.026
520         315      320      14       0.226            0.862       0.005


## O controle: um par sorteado independente

In [4]:
# O controle: um par sorteado independente, onde a razao tem de dar um.
sorteio = np.random.default_rng(SEMENTE)
indice = retornos_a.index.intersection(retornos_b.index)
falso_a = pd.Series(sorteio.normal(0.0, 0.01, len(indice)), index=indice)
falso_b = pd.Series(sorteio.normal(0.0, 0.01, len(indice)), index=indice)
controle = dependencia.juntos(dependencia.rompimentos(falso_a, JANELA, CAUDA),
                              dependencia.rompimentos(falso_b, JANELA, CAUDA))
print("par sorteado: rompe a %.3f%% | rompe b %.3f%% | juntos %d = %.3f%% | esperado %.3f%% | razao %.2fx"
      % (100 * controle["taxa_a"], 100 * controle["taxa_b"], controle["juntos"],
         100 * controle["taxa_juntos"], 100 * controle["esperado"], controle["excesso"]))

par sorteado: rompe a 5.260% | rompe b 5.211% | juntos 19 = 0.307% | esperado 0.274% | razao 1.12x


## O que a proteção individual não cobre

In [5]:
# O que a protecao individual nao cobre: a carteira no dia conjunto e no dia de uma perna so.
carteira = dependencia.carteira(retornos_a, retornos_b, PESO)
em_a = rompe_a.reindex(carteira.index).fillna(False).astype(bool)
em_b = rompe_b.reindex(carteira.index).fillna(False).astype(bool)
juntos = em_a & em_b
separados = em_a ^ em_b
perda_juntos = dependencia.perda_media(carteira, juntos)
perda_so_uma = dependencia.perda_media(carteira, separados)
print("carteira de peso %.2f: %d dias | %d com as duas rompidas | %d com uma so" % (
    PESO, len(carteira), int(juntos.sum()), int(separados.sum())))
print("perda media: %.4f%% no dia conjunto | %.4f%% no dia de uma perna so | razao %.2f" % (
    100 * perda_juntos, 100 * perda_so_uma, perda_juntos / perda_so_uma))

carteira de peso 0.50: 6450 dias | 115 com as duas rompidas | 402 com uma so
perda media: -3.7314% no dia conjunto | -1.7087% no dia de uma perna so | razao 2.18


## As figuras

In [6]:
# Figura 1: o par, dia a dia. Os pontos vermelhos sao os dias em que as duas romperam.
a = retornos_a.reindex(carteira.index)
b = retornos_b.reindex(carteira.index)
fig, eixo = plt.subplots(figsize=(6.6, 6.0))
eixo.plot(100 * a[~juntos], 100 * b[~juntos], ".", color="#9aa5b1", ms=4, label="dia comum")
eixo.plot(100 * a[juntos], 100 * b[juntos], ".", color="#b03a2e", ms=7,
          label="as duas romperam (%d dias)" % int(juntos.sum()))
eixo.axhline(0, color="#7f7f7f", lw=0.8)
eixo.axvline(0, color="#7f7f7f", lw=0.8)
eixo.set_xlabel("retorno de %s no dia (%%)" % SERIE_A)
eixo.set_ylabel("retorno de %s no dia (%%)" % SERIE_B)
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
eixo.set_aspect("equal")
fig.tight_layout()
graficos.salvar(fig, "E05_dependencia", 1)
plt.close(fig)
print("nuvem: %d dias, %d deles com as duas rompidas" % (len(a), int(juntos.sum())))

nuvem: 6450 dias, 115 deles com as duas rompidas


In [7]:
# Figura 2: a razao contra a independencia, com o par real e com o par trocado.
fig, eixo = plt.subplots(figsize=(8.4, 4.2))
x = np.arange(len(tabela))
eixo.bar(x, tabela["x independencia"].to_numpy(), 0.55, color="#1f4e79")
eixo.axhline(1.0, color="#b03a2e", ls="--", lw=1.4, label="o que a independencia prevê: 1")
for i, v in enumerate(tabela["x independencia"].to_numpy()):
    eixo.annotate("%.1fx" % v, (i, v), textcoords="offset points", xytext=(0, 4),
                  ha="center", fontsize=9)
eixo.set_xticks(x)
eixo.set_xticklabels(["o par real" if a == 0 else "atraso de %d dia%s" % (a, "" if a == 1 else "s") for a in tabela.index],
                     fontsize=9)
eixo.set_ylabel("juntos, contra a independência")
eixo.set_ylim(0, max(tabela["x independencia"]) * 1.2)
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E05_dependencia", 2)
plt.close(fig)
print("razoes:", [round(v, 2) for v in tabela["x independencia"]])

razoes: [7.17, 1.43, 1.2, 0.67, 0.86]


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Uma nuvem de seis mil e quatrocentos dias, com o retorno de um mercado no eixo
horizontal e o do outro no vertical. A nuvem cinza é uma elipse esticada do canto de baixo à
esquerda para o de cima à direita, e as duas linhas em zero partem o plano em quatro quadrantes.
Os cento e quinze pontos vermelhos estão **todos** no quadrante de baixo à esquerda — as duas
quedas no mesmo dia —, e nenhum aparece em qualquer outro lugar do plano. O que a figura engana:
as linhas de zero não são o corte de nenhuma das pernas; o corte de cada dia é diferente e fica
bem mais fundo que o zero, de modo que o quadrante vermelho é bem maior do que os pontos
vermelhos, e quem olha depressa lê "todo dia de queda" onde a figura está dizendo "os dois
fundos no mesmo dia".

**Figura 2.** Cinco barras azuis e uma linha tracejada em um. A barra do par real é a mais alta
de todas, e as quatro seguintes — o mesmo par com o segundo mercado deslocado em um dia, três
semanas, nove meses e um ano e meio — encostam na linha da independência. O que o eixo engana: os
atrasos estão espaçados por igual no desenho e no tempo não estão, o que faz a queda parecer
suave e contínua; o que a figura mostra, lida com o tempo de verdade, é que **o primeiro passo
já basta** — quase toda a dependência morre com um único dia de deslocamento. A linha do um
também é um chão que não existe no dado: a barra do ano fica abaixo dela, e estar abaixo de um
não quer dizer dependência negativa, quer dizer apenas contagem pequena.


In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "dependencia_dias": par["dias"],
    "dependencia_rompe_a": par["rompe_a"],
    "dependencia_rompe_b": par["rompe_b"],
    "dependencia_taxa_a_pct": 100 * par["taxa_a"],
    "dependencia_taxa_b_pct": 100 * par["taxa_b"],
    "dependencia_juntos": par["juntos"],
    "dependencia_juntos_pct": 100 * par["taxa_juntos"],
    "dependencia_esperado_pct": 100 * par["esperado"],
    "dependencia_esperado_dias": par["esperado_dias"],
    "dependencia_excesso": par["excesso"],
    "dependencia_acompanhados_pct": 100 * par["acompanhados"],
    "dependencia_acompanhados_independente_pct": 100 * par["taxa_b"],
    "dependencia_correlacao": float(tabela.loc[0, "correlacao"]),
    "dependencia_controle_excesso": controle["excesso"],
    "dependencia_perda_juntos_pct": 100 * perda_juntos,
    "dependencia_perda_uma_perna_pct": 100 * perda_so_uma,
    "dependencia_perda_razao": perda_juntos / perda_so_uma,
}
for atraso in ATRASOS:
    extenso = {1: "um_dia", 21: "vinte_e_um_dias", 260: "um_ano", 520: "dois_anos"}[atraso]
    resultado["dependencia_atraso_%s_excesso" % extenso] = float(tabela.loc[atraso, "x independencia"])
    resultado["dependencia_atraso_%s_correlacao" % extenso] = float(tabela.loc[atraso, "correlacao"])


# O que sai do laboratorio e o que o livro cita: medida que o livro nao usa e medida morta.
CITADAS_NO_LIVRO = ("dependencia_acompanhados_independente_pct", "dependencia_acompanhados_pct", "dependencia_atraso_dois_anos_excesso", "dependencia_atraso_um_ano_excesso", "dependencia_atraso_um_dia_correlacao", "dependencia_atraso_um_dia_excesso", "dependencia_atraso_vinte_e_um_dias_excesso", "dependencia_controle_excesso", "dependencia_correlacao", "dependencia_dias", "dependencia_esperado_dias", "dependencia_esperado_pct", "dependencia_excesso", "dependencia_juntos", "dependencia_juntos_pct", "dependencia_perda_juntos_pct", "dependencia_perda_razao", "dependencia_perda_uma_perna_pct", "dependencia_rompe_a", "dependencia_rompe_b", "dependencia_taxa_a_pct", "dependencia_taxa_b_pct", )
resultado = {chave: valor for chave, valor in resultado.items() if chave in CITADAS_NO_LIVRO}

caminho = Path("lab/resultados/E05_dependencia.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E05_dependencia.json gravado | 22 grandezas
